#### get text content from data
If you use this dataset, please cite the following paper:
Urchs et al. (2025): taz2024full: Analysing German Newspapers for Gender Bias and Discrimination across Decades, Findings of ACL 2025.

format:
- meta data:

    - "published_on": publishing date in the format YYYY-MM-DDThh:mm:ss+01:00

    - "contains_actors": boolean indicating whether person entities were detected

    - "crawled_on": crawling date

    - "language": always "de"

    - "type": always "article"

    - "author": article author

    - "keywords": topic keywords

    - "token_count": number of tokens


- text:

    - "title": article title

    - "teaser": short summary

    - "text": main body of the article

In [1]:
import pandas as pd
import ast
import json
from pprint import pprint
import numpy as np
import os
import re
from io import StringIO
from tqdm import tqdm
import heapq
from collections import Counter
import string
from sklearn.feature_extraction.text import TfidfVectorizer

import spacy
from spacy.language import Language
from spacy_language_detection import LanguageDetector

nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm


def flatten(xss):
    return [x for xs in xss for x in xs]

In [2]:
c_directory = r"corpus" 

# Iterate over files in directory
dict_year = {}
for name in os.listdir(c_directory):
    dir = os.path.join(c_directory, name)
    for n in os.listdir(dir):
        with open(os.path.join(dir, n), encoding="utf-8") as f:
            data = json.load(f)
            dict_year[name] = data
print(sorted(dict_year.keys()))

['1980', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [3]:
# separate sentencesy with spacy
def separate_sentences(sents):
    doc = nlp(sents)
    res = []
    for sent in doc.sents:
        if not re.search(r'(hb|hh$)|(/^h\s+hb\s+hh$)+\ski?', sent.text.lower()):
            res.append(sent.text.lower())
    return res

In [4]:
# language detection
def get_lang_detector(nlp, name):
    return LanguageDetector(seed=42)


def detect_language(text):
    try:
        Language.factory("language_detector", func=get_lang_detector)
        nlp.add_pipe('language_detector', last=True)
    except:
        pass
    languages = []
    # for sentences
    if text.count(".") > 1: 
        # Document level language detection
        doc = nlp(text)
        languages.append(doc._.language['language'])
    #for full text
    else:
        doc = nlp(text)
        for i, sent in enumerate(doc.sents):
            #print(sent, sent._.language)
            languages.append(sent._.language['language'])

    #print(languages)
    return languages

In [5]:
# keep text content and id only, divide text into sentences
def save_sentences_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in tqdm(dict_year.keys()):
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            if "de" in detect_language(text):
                text_dict[k] = text
            with open(f"{text_dir}{k}.txt", "w") as f:
                    f.write(json.dumps(separate_sentences(str(text))))

# keep text content and id only
def save_txt_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in tqdm(dict_year.keys()):
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            if "de" in detect_language(text):
                text_dict[k] = text
            if not re.search(r'(hb|hh$)|(/^h\s+hb\s+hh$)+\ski?', text):
                text_dict[k] = text
            with open(f"{text_dir}{k}_full.txt", "w") as f:
                    f.write(str(text))

save_sentences_to_file()
save_txt_to_file()

Directory 'corpus_texts/' already exists.


100%|██████████| 43/43 [6:23:46<00:00, 535.49s/it]  


Directory 'corpus_texts/' already exists.


100%|██████████| 43/43 [3:10:25<00:00, 265.70s/it]  


#### get sentences with occurences of AI related words
Notes: 
- Englisch and German for certain terms?
- Specific product names
- ML/AI system terminology
- Nouns only
- Abreviations, too



steps:
- build AI vocabulary
- filter by sentence containing at least one AI vocab
- multiply sentences if they contain multiple terms 

In [6]:
# AI-terms from ANTHROSCORE paper, transalted with deepL
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
print(ki_terms)
regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

['ki', 'künstliche intelligenz', 'chatgpt', 'ai', 'copilot', 'gemini', 'chatbot', 'llm', 'chat-gpt', 'sprachmodell', 'ki-modell', 'ki-system', 'ki-tool', 'artificial intelligence', 'ki-anwendung', 'palm', 'lms', 'llama', 'transformer', 'sprachmodelle', 'sprachmodell', 'gpt', 'plms', 'vortrainierte sprachmodelle', 'gpt-2', 'xlnet', 'große sprachmodelle', 'llms', 'gpt-3', 'stiftungsmodell', 'gpt-neo', 'gpt-j', 'chatgpt', 'gpt-4']
\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_2429/3726958728.py:3: SyntaxWarning: invalid escape sequence '\s'
  regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_2429/3726958728.py:11: SyntaxWarning: invalid escape sequence '\s'
  regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]


#### Sätze säubern, filtern
- Satzzeichen
- (Html, etc)
- POS (use spacy bc TIGER corpus trained)
- Lexikon für Verb- / Adjektivtypen

In [7]:
# preprocessing auxiliary functions

stopwords_ger = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords_ger = ast.literal_eval(stopwords_ger)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus, stopwords):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out

def remove_stop_words(token_list):
    return [token for token in token_list if not token.is_stop]

    
def lemmatizing(token_list):
    return [token.lemma_ for token in token_list]
    
def get_POS(token_list):
    return [token.pos_ for token in token_list]

In [8]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
# stopwords_l a list of stopwords in the target language if not english
def calc_tf_idf(corpus, stopwords_l= None):
    corpus_cleaned = [remove_punctuation(doc.lower()) for doc in corpus]
    if stopwords_l:
        corpus_cleaned = [remove_stop_words_from_text(doc.split(), stopwords_l) for doc in corpus_cleaned]
        vectorizer = TfidfVectorizer(stop_words=stopwords_l, lowercase=True, ngram_range=(1,2))
    else:
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1,2))
    # fit tfidf
    tfidf_m = vectorizer.fit_transform(corpus_cleaned) #transform
    #print(tfidf_m.shape)
    feature_array = vectorizer.get_feature_names_out()
    # top (20?) for every doc
    tfidf_sorting = np.argsort(tfidf_m.toarray()).flatten()[::-1] 
    top_n = feature_array[tfidf_sorting].tolist()[:20]
    return top_n #array of list(s?)

#### get sentences with ki_terms

In [9]:
# filter ki_terms
def filter_full_ki_terms_regex(c_dir,regex,path):
    ki_sentences = {}
    try:
        for dir in tqdm(os.listdir(c_dir)):
            sent_l = []
            if dir.endswith("_full.txt"):
                sent_l = open(os.path.join(c_dir,dir)).read()
                # could also account for lemmata, etc
                sent_l = remove_punctuation(sent_l)        
                # filter
                res_tmp = []
                if re.search(regex, sent_l):
                    #print("res",sent, re.search(regex_ki, sent).string)
                    res_tmp.append(sent_l)
                # add to dict
                if len(res_tmp) > 0:
                    ki_sentences [dir] =  res_tmp
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = remove_punctuation(elem["translation"])
            # filter
            res_tmp = []
            if re.search(regex, sent):
                #print("res",sent, re.search(regex_ki, sent).string)
                res_tmp.append(sent)
            # add to dict
            if len(res_tmp) > 0:
                ki_sentences [elem["id"]] =  res_tmp

    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)
    
    
def filter_ki_terms_regex(c_dir,regex,path):
    ki_sentences = {}
    try:
        for dir in tqdm(os.listdir(c_dir)):
            if not dir.endswith("_full.txt"):
                with open(os.path.join(c_dir,dir)) as f:
                    sent_l = json.load(f)
                    # could also account for lemmata, etc
                    sent_l = [remove_punctuation(sent) for sent in sent_l]   
                #filter
                res_tmp = []
                for sent in sent_l:
                    if re.search(regex, sent):
                    #print("res",sent, re.search(regex_ki, sent).st‚ring)
                        res_tmp.append(sent)
                    # add to dict
                    if len(res_tmp) > 0:
                        ki_sentences [dir] =  res_tmp
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = remove_punctuation(elem["translation"])
            #filter
            res_tmp = []
            if re.search(regex, sent):
            #print("res",sent, re.search(regex_ki, sent).st‚ring)
                res_tmp.append(sent)
            # add to dict
            if len(res_tmp) > 0:
                ki_sentences [elem["id"]] =  res_tmp
            
    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)

In [10]:
# filter using TF-IDF instead of all tokens
# function filter_full_ki_terms_tfidf() following not useful since tfidf needs sentence list as corpus

def filter_ki_terms_tfidf(c_dir,terms,path,stopwords_l= None):
    ki_sentences = {}
    try: 
        for dir in tqdm(os.listdir(c_dir)):
            if not dir.endswith("_full.txt"):
                #sent_l = []
                with open(os.path.join(c_dir,dir)) as f:
                    sent_l = json.load(f)
                    # could also account for lemmata, etc
                    sent_l = [remove_punctuation(sent) for sent in sent_l]
                # compare tf-idf with ki_terms here
                if len(sent_l) > 1:
                    tfidf_words = calc_tf_idf(sent_l,stopwords_l)
                    if any(tfidf_words) in terms:
                        ki_sentences[dir] = {"sentences": str(sent_l), "top_words": str(tfidf_words)}
                    else:
                        ki_sentences[dir] = {"sentences":None, "top_words": str(tfidf_words)}
                else:
                    continue
                    #print(sent_l)
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = [remove_punctuation(elem["translation"])]
            if len(sent[0].split()) > 1:
                tfidf_words = calc_tf_idf(sent,stopwords_l)
                if any(tfidf_words) in terms:
                    ki_sentences[elem["id"]] = {"sentences": elem["text"], "top_words": str(tfidf_words)}
                else:
                    ki_sentences[elem["id"]] = {"sentences": None, "top_words": str(tfidf_words)}
            else:
                continue

    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)

In [11]:
c_txt_dir = r"corpus_texts" 

ki_sentences_json = filter_ki_terms_regex(c_txt_dir, regex_ki,"KI_sentences.json")
ki_sentences_full_json =filter_full_ki_terms_regex(c_txt_dir, regex_ki, "KI_sentences_full.json")
ki_sentences_tfidf_json = filter_ki_terms_tfidf(c_txt_dir, ki_terms, "KI_sentences_tfidf.json", stopwords_ger)

100%|██████████| 306612/306612 [01:02<00:00, 4890.29it/s]


859


100%|██████████| 306612/306612 [01:03<00:00, 4838.06it/s]


150


100%|██████████| 306612/306612 [06:10<00:00, 826.54it/s] 


152134


In [12]:
# filter non-german text from KI_sentences_full
with open("generated_data/KI_sentences_full.json", "r") as f:
    txt_json = dict(json.load(f))

tmp = {}
for key,val in txt_json.items():
    if "de" in detect_language(val[0]):
        tmp[key] = val
print(len(tmp.keys()))

138


In [13]:
# save to file
with open(f'generated_data/KI_sentences_full.json', 'w') as f:
    json.dump(tmp, f)

In [14]:
with open("generated_data/KI_sentences_full.json", "r") as f:
    txt_json = dict(json.load(f))

len(txt_json.keys())

138

##### get AI terms of translations

In [ ]:
c_txt_dir_full_eng = "generated_data/KI_sentences_full_translated.json"
c_txt_dir_eng = "generated_data/KI_sentences_translated.json"

#ai_sentences_json = filter_ki_terms_regex(c_txt_dir_eng, regex_ai, "AI_sentences.json")
#ai_sentences_full_json = filter_full_ki_terms_regex(c_txt_dir_full_eng, regex_ai, "AI_sentences_full.json")
#ai_sentences_tfidf_json = filter_ki_terms_tfidf(c_txt_dir_eng, ai_terms, "AI_sentences_tfidf.json")


In [23]:
# get translations for filtered data
paths = ["KI_sentences.json","KI_sentences_full.json", "KI_sentences_tfidf.json"]
t_paths = ["AI_sentences.json", "AI_sentences_full.json", "AI_sentences_tfidf.json"]
                                                
def get_translations(file_path, tanslation_path):
    # get file content
    data = pd.read_json(StringIO(open(f'generated_data/{file_path}').read()),orient='records')
    ids = data['id']
    print("d",ids)
    # get tranlsation content
    transl = pd.read_json(StringIO(open(f'generated_data/{tanslation_path}').read()))
    t_ids = transl['id']
    print("t",ids)

    res = [x for x in ids if x in t_ids]
    mask = transl['id'].isin(res)
    return transl.loc[~mask]


In [24]:
df = get_translations(paths[0],t_paths[0])

ValueError: All arrays must be of the same length